# LLM Evaluation — Austrian Tax Law QA
**Project Stage 3 | WU Vienna | SS26**

Evaluates two model outputs against the ground truth using BLEU, ROUGE-1, ROUGE-L and BERTScore.

**Models:**
- Model 1: DeepSeek-V3 (inference only via API)
- Model 2: Qwen2.5-1.5B-Instruct fine-tuned with LoRA (SFT)

**Note on row counts:** The ground truth covers 645 questions. Both model outputs contain 643 answers. However, the model outputs include 81 `VAT-INTL` IDs that are absent from the ground truth. The evaluation therefore runs on **562 matched rows** for each model — the largest set where a ground truth answer exists.

## 1. Install & Import

In [ ]:
# Run once, then comment out
!pip install rouge-score nltk bert-score pandas tqdm matplotlib --quiet

In [ ]:
import csv
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from tqdm import tqdm

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("All imports OK.")

## 2. File Paths

In [ ]:
# Ground truth  — semicolon-separated, UTF-8 BOM
GT_PATH = "./data_superclean.csv"

# DeepSeek output — comma-separated, UTF-8 BOM
M1_PATH  = "./inference_results_deepseek_v3.csv"
M1_LABEL = "DeepSeek-V3"

# SFT (Qwen) output — semicolon-separated, UTF-8 BOM
M2_PATH  = "./sft_results_qwen.csv"
M2_LABEL = "Qwen2.5-1.5B SFT"

## 3. Load Data

Each file has a different separator:
- Ground truth and SFT output → **semicolon** (`;`)
- DeepSeek output → **comma** (`,`), answers may contain commas so we use `csv.reader`

In [ ]:
# --- Ground truth ---
gt = pd.read_csv(GT_PATH, sep=";", encoding="utf-8-sig")
gt["id"] = gt["id"].str.strip()
gt["correct_answer"] = gt["correct_answer"].str.strip()
print(f"Ground truth  : {len(gt)} rows")

# --- DeepSeek (comma-separated; answers may contain commas) ---
rows = []
with open(M1_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        if len(row) >= 2:
            rows.append({"id": row[0].strip(), "answer": ",".join(row[1:]).strip()})
m1 = pd.DataFrame(rows)
print(f"DeepSeek      : {len(m1)} rows")

# --- SFT Qwen (semicolon-separated) ---
rows = []
with open(M2_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f, delimiter=";")
    next(reader)  # skip header
    for row in reader:
        if len(row) >= 2:
            rows.append({"id": row[0].strip(), "answer": ";".join(row[1:]).strip()})
m2 = pd.DataFrame(rows)
print(f"Qwen SFT      : {len(m2)} rows")

# --- Merge with ground truth ---
df1 = gt.merge(m1, on="id", how="inner")
df2 = gt.merge(m2, on="id", how="inner")
print(f"\nMatched rows for evaluation:")
print(f"  DeepSeek : {len(df1)}")
print(f"  Qwen SFT : {len(df2)}")
print(f"\n(Unmatched rows are VAT-INTL IDs missing from the ground truth — excluded from evaluation.)")

## 4. Compute Metrics

In [ ]:
smoother = SmoothingFunction().method1
rouge    = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)

def evaluate(df, label):
    print(f"\nEvaluating {label} ({len(df)} rows)...")
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        ref = str(row["correct_answer"])
        hyp = str(row["answer"])

        # BLEU
        bleu = sentence_bleu(
            [nltk.word_tokenize(ref.lower())],
            nltk.word_tokenize(hyp.lower()),
            smoothing_function=smoother
        )

        # ROUGE
        r = rouge.score(ref, hyp)

        results.append({
            "id"     : row["id"],
            "bleu"   : round(bleu, 4),
            "rouge1" : round(r["rouge1"].fmeasure, 4),
            "rougeL" : round(r["rougeL"].fmeasure, 4),
        })

    scores = pd.DataFrame(results)

    # BERTScore (batched — much faster than row-by-row)
    print("  Running BERTScore...")
    P, R, F1 = bert_score(
        df["answer"].tolist(),
        df["correct_answer"].tolist(),
        model_type="bert-base-multilingual-cased",
        batch_size=32,
        verbose=False
    )
    scores["bertscore_f1"] = F1.numpy().round(4)

    print(f"  Done. Mean scores:")
    for col in ["bleu", "rouge1", "rougeL", "bertscore_f1"]:
        print(f"    {col:<14}: {scores[col].mean():.4f}")

    return scores


scores1 = evaluate(df1, M1_LABEL)
scores2 = evaluate(df2, M2_LABEL)

## 5. Results Table

In [ ]:
metrics = ["bleu", "rouge1", "rougeL", "bertscore_f1"]

summary = pd.DataFrame([
    {"Model": M1_LABEL, **scores1[metrics].mean().round(4).to_dict()},
    {"Model": M2_LABEL, **scores2[metrics].mean().round(4).to_dict()},
]).set_index("Model")

print("Average scores across all matched questions")
print("=" * 60)
print(summary.to_string())
print("=" * 60)

# Styled table for Jupyter
summary.style.format("{:.4f}").highlight_max(axis=0, color="#c6efce").set_caption("Higher = better (green = best per metric)")

## 6. Plot

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle("Score Distribution per Model", fontsize=12, fontweight="bold")

colors = ["#4C72B0", "#DD8452"]

for ax, metric in zip(axes, metrics):
    bp = ax.boxplot(
        [scores1[metric].values, scores2[metric].values],
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2)
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(metric.upper())
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["M1", "M2"], fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

handles = [plt.Rectangle((0,0),1,1, fc=c, alpha=0.7) for c in colors]
fig.legend(handles, [M1_LABEL, M2_LABEL], loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig("evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Error Analysis — Worst Examples

In [ ]:
N = 5  # number of worst examples to show

for scores, df, label in [(scores1, df1, M1_LABEL), (scores2, df2, M2_LABEL)]:
    print(f"\n{'='*65}")
    print(f"  {label} — {N} worst examples by BERTScore F1")
    print(f"{'='*65}")
    worst_ids = scores.nsmallest(N, "bertscore_f1")["id"].values
    for qid in worst_ids:
        row   = df[df["id"] == qid].iloc[0]
        score = scores[scores["id"] == qid]["bertscore_f1"].iloc[0]
        print(f"\n  ID           : {qid}")
        print(f"  BERTScore F1 : {score:.4f}")
        print(f"  Reference    : {str(row['correct_answer'])[:200]}")
        print(f"  Model answer : {str(row['answer'])[:200]}")

## 8. Save Results

In [ ]:
scores1["model"] = M1_LABEL
scores2["model"] = M2_LABEL

all_scores = pd.concat([scores1, scores2], ignore_index=True)
all_scores.to_csv("evaluation_scores.csv", index=False, encoding="utf-8-sig")
summary.to_csv("evaluation_summary.csv", encoding="utf-8-sig")

print("Saved: evaluation_scores.csv")
print("Saved: evaluation_summary.csv")